# SnapCal Colab Training Notebook

This notebook downloads Food-101 from Kaggle, runs the SnapCal preprocessing and training pipeline, and exports a `production_bundle` that can be deployed with the Django API on Render.

Before you run it:
- Push this repo to GitHub and replace `REPO_URL` below.
- Switch Colab to a GPU runtime.
- Make sure your `food101_usda_mapping.csv` has been curated before you trust calorie metrics.
- This version validates the repo clone, auto-detects the extracted Food-101 root, and runs every script from the repo root so Colab cwd drift does not break training.


In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/SnapCal.git"
REPO_BRANCH = "main"
GITHUB_TOKEN = None  # Set this to a GitHub PAT if the repo is private.
KAGGLE_DATASET = "dansbecker/food-101"
RUN_NAME = "snapcal_v1"
DRIVE_ROOT = "/content/drive/MyDrive/snapcal_runs"
REPO_DIR = "/content/SnapCal"
DATA_ROOT = "/content/data/raw"
EXPORT_ROOT = f"{DRIVE_ROOT}/{RUN_NAME}"
MOBILE_SAM_CHECKPOINT_URL = "https://github.com/ChaoningZhang/MobileSAM/raw/master/weights/mobile_sam.pt"
FOOD101_ROOT = None
EXPORT_CONFIG = "configs/training/vit_b16_segmented.json"
EXPORT_CHECKPOINT = "artifacts/models/vit_b16_segmented/best.pt"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os
from pathlib import Path
import shlex
import shutil
import subprocess
from urllib.parse import quote
import zipfile

def run_checked(args, cwd=None, display_args=None):
    args = [str(arg) for arg in args]
    visible_args = args if display_args is None else [str(arg) for arg in display_args]
    printable = ' '.join(shlex.quote(arg) for arg in visible_args)
    print(f'$ {printable}')
    subprocess.run(args, check=True, cwd=None if cwd is None else str(cwd))

def build_clone_url():
    if not GITHUB_TOKEN:
        return REPO_URL
    if not REPO_URL.startswith('https://github.com/'):
        raise ValueError('GITHUB_TOKEN support is only implemented for https://github.com URLs.')
    return f"https://x-access-token:{quote(GITHUB_TOKEN, safe='')}@{REPO_URL[len('https://') :]}"

def detect_food101_root(search_root):
    search_root = Path(search_root)
    matches = []
    for train_file in search_root.rglob('meta/train.txt'):
        candidate = train_file.parent.parent
        if (candidate / 'meta' / 'test.txt').exists() and (candidate / 'images').exists():
            matches.append(candidate)
    if not matches:
        raise FileNotFoundError(
            f'Could not find a Food-101 dataset root under {search_root}. ' 
            'Expected a directory containing both meta/train.txt and images/.'
        )
    unique_matches = sorted(set(matches), key=lambda path: (len(path.parts), str(path)))
    return unique_matches[0]

def ensure_repo_cloned():
    if 'YOUR_USERNAME' in REPO_URL:
        raise ValueError('Set REPO_URL to your actual GitHub repository before running this cell.')
    repo_path = Path(REPO_DIR)
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
    if repo_path.exists():
        shutil.rmtree(repo_path)
    clone_url = build_clone_url()
    try:
        run_checked(
            ['git', 'clone', '--branch', REPO_BRANCH, clone_url, REPO_DIR],
            display_args=['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, REPO_DIR],
        )
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Git clone failed. Verify that REPO_URL exists and REPO_BRANCH is correct. ' 
            'If the repository is private, set GITHUB_TOKEN to a GitHub personal access token with repo read access.'
        ) from exc
    expected = repo_path / 'ml' / 'scripts' / 'train.py'
    if not expected.exists():
        raise FileNotFoundError(f'Clone finished but expected file is missing: {expected}')
    return repo_path

repo_path = ensure_repo_cloned()
print(f'Repo cloned to {repo_path}')


In [ ]:
run_checked(['python', '--version'])
run_checked(['python', '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'])
run_checked([
    'python', '-m', 'pip', 'install',
    'kaggle',
    'numpy>=1.26,<3.0',
    'Pillow>=10.4,<11.0',
    'matplotlib>=3.8,<4.0',
    'pandas>=2.2,<3.0',
    'scikit-learn>=1.5,<2.0',
    'transformers>=4.45,<5.0',
])
try:
    import torch
    import torchvision
except ImportError:
    run_checked(['python', '-m', 'pip', 'install', 'torch>=2.4,<3.0', 'torchvision>=0.19,<1.0'])
    import torch
    import torchvision

print(f'torch={torch.__version__}')
print(f'torchvision={torchvision.__version__}')
print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
else:
    print('GPU is not available. Switch Colab to a GPU runtime before continuing.')

run_checked(['python', '-m', 'pip', 'install', 'git+https://github.com/ChaoningZhang/MobileSAM.git'])
run_checked(['python', '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=REPO_DIR)
Path(REPO_DIR, 'artifacts', 'cache').mkdir(parents=True, exist_ok=True)
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
run_checked(['wget', '-O', str(Path(REPO_DIR) / 'artifacts' / 'cache' / 'mobile_sam.pt'), MOBILE_SAM_CHECKPOINT_URL])


In [ ]:
from getpass import getpass

kaggle_username = input('Kaggle username: ').strip()
kaggle_key = getpass('Kaggle key: ').strip()

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
credentials_path = kaggle_dir / 'kaggle.json'
credentials_path.write_text(json.dumps({'username': kaggle_username, 'key': kaggle_key}), encoding='utf-8')
os.chmod(credentials_path, 0o600)

archive_path = Path(DATA_ROOT) / 'food-101.zip'
kaggle_extract_succeeded = False
for attempt in range(2):
    for stale_name in ('food-101', '__MACOSX'):
        stale_path = Path(DATA_ROOT) / stale_name
        if stale_path.exists():
            shutil.rmtree(stale_path)
    if archive_path.exists():
        archive_path.unlink()
    run_checked(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', DATA_ROOT, '--force'])
    if not archive_path.exists():
        raise FileNotFoundError(f'Kaggle download finished but archive is missing: {archive_path}')
    print(f'Downloaded archive size: {archive_path.stat().st_size / (1024 ** 3):.2f} GiB')
    try:
        run_checked(['unzip', '-q', '-o', str(archive_path), '-d', DATA_ROOT])
        kaggle_extract_succeeded = True
        break
    except subprocess.CalledProcessError:
        if attempt == 1:
            print('Kaggle archive extraction failed twice; falling back to the official Food-101 download from torchvision.')
        else:
            print('Archive extract failed; deleting the zip and retrying download once...')

if not kaggle_extract_succeeded:
    for stale_name in ('food-101', '__MACOSX'):
        stale_path = Path(DATA_ROOT) / stale_name
        if stale_path.exists():
            shutil.rmtree(stale_path)
    if archive_path.exists():
        archive_path.unlink()
    from torchvision.datasets import Food101
    Food101(root=DATA_ROOT, split='train', download=True)

FOOD101_ROOT = str(detect_food101_root(DATA_ROOT))
print(f'Resolved FOOD101_ROOT = {FOOD101_ROOT}')
for path in sorted(Path(FOOD101_ROOT).iterdir()):
    print(path)


In [ ]:
import time

manifest_path = Path(REPO_DIR) / 'data' / 'reference' / 'manifests' / 'food101_manifest.csv'
print(f'Manifest will be written to: {manifest_path}')

print('1/4 Verifying CUDA availability...')
print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('CUDA is not available. Switch Colab to a GPU runtime before running segmentation.')

print('2/4 Building manifest...')
t0 = time.time()
run_checked([
    'python',
    'ml/scripts/build_manifest.py',
    '--dataset-root',
    FOOD101_ROOT,
    '--output',
    'data/reference/manifests/food101_manifest.csv',
], cwd=REPO_DIR)
print(f'Manifest step finished in {(time.time() - t0) / 60:.2f} min')

if not manifest_path.exists():
    raise FileNotFoundError(f'Manifest was not created: {manifest_path}')

print('3/4 Running smoke test...')
t0 = time.time()
run_checked(['python', 'ml/scripts/smoke_test.py', '--config', 'configs/training/resnet50_raw.json'], cwd=REPO_DIR)
print(f'Smoke test finished in {(time.time() - t0) / 60:.2f} min')

print('4/4 Running segmentation sample (10 images)...')
t0 = time.time()
run_checked([
    'python',
    'ml/scripts/run_segmentation.py',
    '--manifest',
    'data/reference/manifests/food101_manifest.csv',
    '--config',
    'configs/segmentation/mobilesam_default.json',
    '--split',
    'train',
    '--limit',
    '10',
], cwd=REPO_DIR)
print(f'Segmentation sample finished in {(time.time() - t0) / 60:.2f} min')


## Full Segmentation

Run this only after the smoke test and the 10-image segmentation sanity check succeed. It precomputes segmented crops and masks for all splits.


In [ ]:
run_checked([
    'python',
    'ml/scripts/run_segmentation.py',
    '--manifest',
    'data/reference/manifests/food101_manifest.csv',
    '--config',
    'configs/segmentation/mobilesam_default.json',
    '--split',
    'train',
], cwd=REPO_DIR)
run_checked([
    'python',
    'ml/scripts/run_segmentation.py',
    '--manifest',
    'data/reference/manifests/food101_manifest.csv',
    '--config',
    'configs/segmentation/mobilesam_default.json',
    '--split',
    'val',
], cwd=REPO_DIR)
run_checked([
    'python',
    'ml/scripts/run_segmentation.py',
    '--manifest',
    'data/reference/manifests/food101_manifest.csv',
    '--config',
    'configs/segmentation/mobilesam_default.json',
    '--split',
    'test',
], cwd=REPO_DIR)


## Training Runs

The raw-image baselines should run first. Then repeat the same order for the segmented datasets. Every command below runs from `REPO_DIR`, so it does not depend on the notebook's current working directory.


In [ ]:
raw_runs = [
    'configs/training/resnet50_raw.json',
    'configs/training/efficientnet_b0_raw.json',
    'configs/training/vit_b16_raw.json',
]
segmented_runs = [
    'configs/training/resnet50_segmented.json',
    'configs/training/efficientnet_b0_segmented.json',
    'configs/training/vit_b16_segmented.json',
]
training_runs = raw_runs + segmented_runs

for config_path in training_runs:
    config_file = Path(REPO_DIR) / config_path
    if not config_file.exists():
        raise FileNotFoundError(f'Missing config file: {config_file}')
    print(f'Running training config: {config_path}')
    run_checked(['python', 'ml/scripts/train.py', '--config', config_path], cwd=REPO_DIR)


In [ ]:
checkpoint_path = Path(REPO_DIR) / EXPORT_CHECKPOINT
if not checkpoint_path.exists():
    raise FileNotFoundError(
        f'Expected checkpoint was not found: {checkpoint_path}. ' 
        'Update EXPORT_CHECKPOINT if you exported a different training run.'
    )
run_checked([
    'python',
    'ml/scripts/export_inference_bundle.py',
    '--checkpoint',
    EXPORT_CHECKPOINT,
    '--config',
    EXPORT_CONFIG,
    '--mapping',
    'data/reference/food101_usda_mapping.csv',
    '--output-dir',
    'artifacts/models/production_bundle',
    '--model-version',
    RUN_NAME,
], cwd=REPO_DIR)
Path(EXPORT_ROOT).mkdir(parents=True, exist_ok=True)
run_checked(['rsync', '-av', 'artifacts/models/production_bundle', f'{EXPORT_ROOT}/'], cwd=REPO_DIR)
run_checked(['rsync', '-av', 'artifacts/reports', f'{EXPORT_ROOT}/'], cwd=REPO_DIR)
run_checked(['rsync', '-av', 'data/reference/manifests', f'{EXPORT_ROOT}/'], cwd=REPO_DIR)


## Deploy the Exported Bundle

After this finishes:
- Zip the exported `production_bundle` folder if you want Render to fetch it via `SNAPCAL_MODEL_BUNDLE_URL`.
- Deploy the Django API with `render.yaml`.
- Set `NEXT_PUBLIC_API_BASE_URL` in Vercel to your Render API URL.
